# BLOQUE 1_D5C3V1 – Infraestructura peatonal
## Objetivo
Construir capa consolidada ECONOMICA normalizada por unidad espacial.
## Inputs
- censo.csv
- manzana.shp

## Procesos
- Limpieza
- Homologación de CRS
- Join espacial
- Normalización

## Output
1_D5C3_Movilidad.gpkg
- 1_D5C3V1_Infraestructura_peatonal_mzn.shp
	
	

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from dbfread import DBF
import jenkspy
import json
from jenkspy import jenks_breaks
import time

In [2]:
# Rutas
# LEER INFORMACIÓN CENSAL
Censo_Urbn_ruta = r"C:\Users\alflo\Desktop\DAL_LAB\data_raw\Censos\Censo2020_CEU_jal_csv\TI_MANZANA_EU_14.csv"
manzanas_ruta = r"C:\Users\alflo\Desktop\DAL_LAB\data_raw\Manzanas\14m.shp"

ruta_salida = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_1\1_D5C3_Movilidad.gpkg"

In [3]:
# Cargar los GeoDataFrames
MZNA_2020_df = pd.read_csv(Censo_Urbn_ruta, dtype=str)
gdf_manzanas = gpd.read_file(manzanas_ruta)

## Tratado Censo Entorno

In [4]:
# Filtrar censos
MZNA_2020_df_F = MZNA_2020_df[['ENT', 'MUN', 'LOC', 'AGEB', 'MZA', 'ARBOLES_C', 'ALUMPUB_C', 'BANQUETA_C']].copy()

In [5]:
# Lista de columnas a procesar
campos = ['ARBOLES_C', 'ALUMPUB_C', 'BANQUETA_C']

# Diccionario de reemplazo
replacement_map = {
    "1": 100,
    "2": 40,
    "3": 0,
    "8": 0,
    "9": 0
}

for field in campos:
    if field in MZNA_2020_df_F.columns:
        MZNA_2020_df_F.loc[:, field] = (
            MZNA_2020_df_F[field]
            .replace(replacement_map)
            .where(lambda s: s.isin(replacement_map.values()))
        )

C:\Users\alflo\AppData\Local\Temp\ipykernel_12236\3471855614.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(replacement_map)
C:\Users\alflo\AppData\Local\Temp\ipykernel_12236\3471855614.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(replacement_map)
C:\Users\alflo\AppData\Local\Temp\ipykernel_12236\3471855614.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To o

In [6]:
#Crear CVEGEO
MZNA_2020_df_F['CVEGEO'] = MZNA_2020_df_F['ENT'] + MZNA_2020_df_F['MUN'] + MZNA_2020_df_F['LOC'] + MZNA_2020_df_F['AGEB'] + MZNA_2020_df_F['MZA']

# Renombrar columnas de ITER_2020_df antes del merge para evitar conflictos
gdf_manzanas['CVEGEO'] = gdf_manzanas['CVEGEO'].astype(str)
MZNA_2020_df_F['CVEGEO'] = MZNA_2020_df_F['CVEGEO'].astype(str)

renombres = {
    "ARBOLES_C": "ARB", "ALUMPUB_C": "ALU", "BANQUETA_C": "BAN",
}
MZNA_2020_df_F = MZNA_2020_df_F.rename(columns=renombres)

# Obtener dinámicamente las columnas renombradas para cada DataFrame
columnas_mzna = ['CVEGEO', 'ARB', 'ALU', 'BAN']

# Realizar los merges
gdf_manzanas = gdf_manzanas.merge(
    MZNA_2020_df_F[columnas_mzna],
    on='CVEGEO', 
    how='left'
)

In [7]:
gdf_manzanas_fill = gdf_manzanas.fillna(0)

C:\Users\alflo\AppData\Local\Temp\ipykernel_12236\3412677076.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_manzanas_fill = gdf_manzanas.fillna(0)


In [8]:
# Guardar el GeoPackage base actualizado
gdf_manzanas_fill.to_file(ruta_salida, layer='1_D5C3V1_Infraestructura_peatonal_mzn', driver="GPKG")